# U06 · Aprendizaje no supervisado — fenotipado y anomalías

> **Para quien pone el criterio clínico, no el código.** Hasta ahora los datos nos daban la respuesta correcta (`evento_cv`, `riesgo_cv_10a`) y aprendíamos a predecirla. Aquí cambiamos las reglas: trabajamos **sin etiqueta**. Dejamos que los datos **sugieran** grupos de pacientes (fenotipos), agrupamos centros por su perfil, "dibujamos" cohortes con **PCA** y levantamos **alertas** ante casos raros.

> ⚠️ Todos los datos son **sintéticos** (generados por código, semilla fija). **No representan pacientes reales.** La primera celda los crea sola: no hay que descargar nada.

**La gran diferencia con lo anterior:** en supervisado medíamos el acierto contra una verdad conocida. En no supervisado **no hay verdad de referencia**. Eso convierte la **interpretación clínica** en la pieza decisiva: es lo único que separa un hallazgo real de un espejismo estadístico.

**Qué haremos, en clave clínica:**
1. **Fenotipar pacientes** (`pacientes.csv`) con *k-means*, **sin usar la etiqueta**: estandarizar, elegir *k*, caracterizar cada grupo y validarlo *a posteriori*.
2. **Dibujar** esos grupos con **PCA** en 2D para ver si de verdad se separan.
3. **Segmentar centros** (`centros.csv`) por su perfil de actividad.
4. **Detectar anomalías** en `wearable.csv` con *Isolation Forest*: días raros que merecen una mirada.

[Abrir en Colab](https://colab.research.google.com/drive/1JlWxl0hzVbrte3E4Z0FXQVfPfQVDYDid)


## 0. Preparamos los datos (esta celda se ejecuta sola)

La primera celda de cada cuaderno del curso **genera los ficheros sintéticos** en la carpeta de trabajo. Ejecútala una vez (▶ o *Mayús+Enter*) y sigue hacia abajo. Recuerda: son datos **inventados**, sirven para aprender, no para decidir sobre personas.


In [ ]:
# === Generación de los datos sintéticos del curso (se ejecuta sola) ===
# TODOS LOS DATOS SON SINTÉTICOS. No representan pacientes reales. Semilla fija -> reproducible.
import os, numpy as np, pandas as pd

def generar_datos_clinicos(carpeta="."):
    """Crea los CSV del curso si no están ya en la carpeta. Devuelve la ruta."""
    if os.path.exists(os.path.join(carpeta, "pacientes.csv")):
        return carpeta
    RNG = np.random.default_rng(2026)   # misma semilla en todo el curso

    # --- Centros ---
    AREAS = ["Valencia","Alicante","Castellón","Madrid","Barcelona","Sevilla","Bilbao","Zaragoza"]
    nc = 24
    tipo = RNG.choice(["hospital","centro de salud"], nc, p=[0.35,0.65])
    camas = np.where(tipo=="hospital", RNG.integers(120,900,nc), RNG.integers(0,25,nc))
    nserv = np.where(tipo=="hospital", RNG.integers(12,40,nc), RNG.integers(3,10,nc))
    centros = pd.DataFrame({"centro_id":[f"C{i:03d}" for i in range(1,nc+1)],"tipo":tipo,
        "area":RNG.choice(AREAS,nc),"camas":camas,"n_servicios":nserv,
        "urgencias_dia_media":np.where(tipo=="hospital",RNG.integers(80,320,nc),RNG.integers(5,40,nc)),
        "ratio_mayores65":RNG.uniform(0.12,0.34,nc).round(3)})
    centros.to_csv(os.path.join(carpeta,"centros.csv"), index=False)

    # --- Pacientes (tabla central) ---
    N = 20000
    sexo = RNG.choice(["M","F"], N, p=[0.49,0.51])
    edad = np.clip(np.where(RNG.random(N)<0.6, RNG.normal(58,15,N), RNG.normal(40,12,N)),18,95).round().astype(int)
    p_act = np.clip(0.28-0.0016*(edad-18),0.06,0.28); p_ex = np.clip(0.10+0.004*(edad-18),0.10,0.40); p_nu = 1-p_act-p_ex
    tabaquismo = np.array([RNG.choice(["nunca","ex","activo"],p=[a,b,c]) for a,b,c in zip(p_nu,p_ex,p_act)])
    actividad = RNG.choice(["baja","media","alta"], N, p=[0.4,0.4,0.2])
    antec = RNG.binomial(1,0.22,N)
    actn = pd.Series(actividad).map({"baja":1.5,"media":0.0,"alta":-1.3}).values
    imc = np.clip(RNG.normal(26.5,4.2,N)+0.03*(edad-50)+actn+RNG.normal(0,1.0,N),15,48).round(1)
    fa = (tabaquismo=="activo").astype(float); fe = (tabaquismo=="ex").astype(float)
    ta_s = np.clip(RNG.normal(120,12,N)+0.45*(edad-45)+0.7*(imc-25)+6*fa+RNG.normal(0,6,N),85,220).round().astype(int)
    ta_d = np.clip(0.55*ta_s+RNG.normal(20,6,N),50,130).round().astype(int)
    glu = np.clip(RNG.normal(100,12,N)+1.0*(imc-25)+0.3*(edad-45)+9*antec+RNG.normal(0,7,N),60,320).round(1)
    diabetes = (glu>126).astype(int)
    col = np.clip(RNG.normal(195,30,N)+0.4*(edad-45)+1.1*(imc-25)+RNG.normal(0,15,N),110,380).round(1)
    hdl = np.clip(RNG.normal(55,12,N)-0.25*(imc-25)+6*(sexo=="F")-5*fa-3*actn+RNG.normal(0,6,N),20,110).round(1)
    z = (-3.1+0.062*(edad-50)+0.019*(ta_s-120)+0.008*(col-190)-0.028*(hdl-55)+0.055*(imc-25)
         +0.85*fa+0.30*fe+0.60*diabetes+0.45*antec+0.55*(sexo=="M")
         +0.012*np.maximum(edad-65,0)*fa+RNG.normal(0,0.35,N))
    p = 1/(1+np.exp(-z))
    riesgo = (100*p).round(1); evento = RNG.binomial(1,p)
    pacientes = pd.DataFrame({"paciente_id":[f"P{i:05d}" for i in range(1,N+1)],"edad":edad,"sexo":sexo,
        "imc":imc,"ta_sistolica":ta_s,"ta_diastolica":ta_d,"glucemia":glu,"colesterol_total":col,"hdl":hdl,
        "tabaquismo":tabaquismo,"actividad_fisica":actividad,"antecedentes_familiares":antec,
        "diabetes":diabetes,"riesgo_cv_10a":riesgo,"evento_cv":evento})
    pacientes.to_csv(os.path.join(carpeta,"pacientes.csv"), index=False)

    # --- Versión "sucia" para EDA ---
    d = pacientes.copy()
    d["sexo"] = d["sexo"].map(lambda s: RNG.choice({"M":["M","m","Masculino","H","Hombre"],"F":["F","f","Femenino","Mujer"]}[s]))
    idx = RNG.choice(d.index,int(N*0.06),replace=False)
    d["glucemia"] = d["glucemia"].astype(object)   # permite mezclar texto (mmol/L con coma) y números
    d.loc[idx,"glucemia"] = [str(round(v/18.0,1)).replace(".",",") for v in d.loc[idx,"glucemia"]]
    prob_na = np.where(pacientes["edad"]<40,0.12,0.04); mask = RNG.random(N)<prob_na
    d.loc[mask,"hdl"] = np.nan
    d.loc[RNG.choice(d.index,int(N*0.05),replace=False),"colesterol_total"] = np.nan
    d.loc[RNG.choice(d.index,25,replace=False),"edad"] = RNG.integers(150,260,25)
    d.loc[RNG.choice(d.index,20,replace=False),"ta_sistolica"] = -RNG.integers(1,90,20)
    d.loc[RNG.choice(d.index,15,replace=False),"imc"] = RNG.uniform(80,200,15).round(1)
    d["colesterol_total"] = d["colesterol_total"].astype(object); d["ta_diastolica"] = d["ta_diastolica"].astype(object)
    d.loc[RNG.choice(d.index,30,replace=False),"colesterol_total"] = "desconocido"
    d.loc[RNG.choice(d.index,20,replace=False),"ta_diastolica"] = "N/D"
    d["tabaquismo"] = d["tabaquismo"].map(lambda s: RNG.choice({"nunca":["nunca","No fumador","NUNCA"],
        "ex":["ex","exfumador","Ex-fumador"],"activo":["activo","Fumador","SI"]}[s]))
    d = pd.concat([d, d.sample(400,random_state=3)], ignore_index=True)
    idx = RNG.choice(d.index,200,replace=False); d.loc[idx,"paciente_id"] = d.loc[idx,"paciente_id"].map(lambda s:" "+s+" ")
    d = d.sample(frac=1,random_state=11).reset_index(drop=True)
    d.to_csv(os.path.join(carpeta,"pacientes_sucio.csv"), index=False)

    # --- Urgencias (serie temporal) ---
    ND = 730; fechas = pd.date_range("2024-01-01", periods=ND, freq="D")
    doy = fechas.dayofyear.values; dow = fechas.dayofweek.values
    fest_set = {(1,1),(1,6),(5,1),(8,15),(10,12),(11,1),(12,6),(12,8),(12,25)}
    festivo = np.array([1 if (f.month,f.day) in fest_set else 0 for f in fechas])
    gripe = np.array([1 if f.month in (12,1,2) else 0 for f in fechas])
    temp = (15+10*np.sin(2*np.pi*(doy-110)/365)+RNG.normal(0,2.5,ND)).round(1)
    est = 1+0.14*np.sin(2*np.pi*(doy-20)/365)
    sem = np.where(dow==0,1.18,np.where(dow>=5,1.10,1.0))
    mu = 120.0*est*sem*(1+0.22*gripe)*(1+0.15*festivo)
    ingresos = RNG.poisson(np.maximum(mu,1)).astype(int)
    pd.DataFrame({"fecha":fechas.strftime("%Y-%m-%d"),"ingresos":ingresos,"festivo":festivo,
        "temporada_gripe":gripe,"temperatura":temp}).to_csv(os.path.join(carpeta,"urgencias_diarias.csv"), index=False)

    # --- Notas clínicas (texto) ---
    plant = {"cardiología":["dolor torácico opresivo de {t} de evolución, irradiado a brazo izquierdo",
        "disnea de esfuerzo progresiva y palpitaciones, antecedente de hipertensión",
        "control de anticoagulación, fibrilación auricular conocida, sin dolor actual",
        "edemas en miembros inferiores y ortopnea, sospecha de insuficiencia cardiaca"],
      "respiratorio":["tos productiva de {t}, fiebre y disnea, auscultación con crepitantes",
        "sibilancias y opresión torácica, antecedente de asma, mala respuesta a inhalador",
        "disnea súbita y dolor pleurítico, saturación disminuida",
        "tos seca persistente y febrícula, contacto con caso respiratorio"],
      "digestivo":["dolor abdominal en fosa iliaca derecha de {t}, náuseas y febrícula",
        "epigastralgia y pirosis, relación con las comidas, sin signos de alarma",
        "diarrea de {t} sin productos patológicos, buen estado general",
        "ictericia y coluria, dolor en hipocondrio derecho"],
      "neurología":["cefalea intensa de inicio súbito, la peor de su vida, con fotofobia",
        "focalidad neurológica de {t}, debilidad en hemicuerpo y disartria",
        "mareo con giro de objetos y vómitos, sin cortejo vegetativo",
        "crisis comicial presenciada, recuperación progresiva del nivel de conciencia"],
      "traumatología":["caída casual con dolor e impotencia funcional en muñeca, deformidad",
        "lumbalgia mecánica de {t} tras esfuerzo, sin déficit neurológico",
        "esguince de tobillo con edema y dificultad para la deambulación",
        "gonalgia y derrame articular tras traumatismo deportivo"]}
    prio = {"cardiología":[0.35,0.45,0.20],"respiratorio":[0.30,0.45,0.25],"digestivo":[0.20,0.50,0.30],
        "neurología":[0.45,0.35,0.20],"traumatología":[0.12,0.48,0.40]}
    tiempos = ["horas","un día","dos días","una semana","varios días"]; esp_list = list(plant.keys()); rows=[]
    for _ in range(3000):
        e = RNG.choice(esp_list); txt = RNG.choice(plant[e]).replace("{t}",RNG.choice(tiempos))
        rows.append((txt,e,RNG.choice(["alta","media","baja"],p=prio[e]),RNG.choice(centros["centro_id"].values)))
    pd.DataFrame(rows,columns=["texto","especialidad","prioridad","centro_id"]).to_csv(os.path.join(carpeta,"notas_clinicas.csv"),index=False)

    # --- Wearable (señal) ---
    rows=[]
    for s in range(1,201):
        fcb=RNG.normal(66,6); pb=RNG.normal(7500,2500); sb=RNG.normal(7.0,0.8); anom=RNG.random()<0.05
        for dia in range(1,31):
            fc=fcb+RNG.normal(0,3); pasos=max(0,pb+RNG.normal(0,1500)); sue=float(np.clip(sb+RNG.normal(0,0.6),3,11))
            if anom and dia in (14,15,16): fc+=RNG.uniform(18,30); sue-=RNG.uniform(1.5,3)
            rows.append((f"S{s:03d}",dia,round(fc,1),int(pasos),round(sue,1)))
    pd.DataFrame(rows,columns=["sujeto_id","dia","fc_reposo","pasos","horas_sueno"]).to_csv(os.path.join(carpeta,"wearable.csv"),index=False)
    return carpeta

generar_datos_clinicos(".")
print("Datos sintéticos listos en la carpeta actual. (Recuerda: son datos SINTÉTICOS, no reales.)")

## 1. Fenotipar pacientes: agrupar sin etiquetas

El **clustering** divide los datos en grupos (**clústeres**) de forma que los elementos de un mismo grupo se parezcan entre sí y se diferencien de los demás. Aplicado a pacientes tiene un nombre propio: **fenotipado** —descubrir subgrupos con un perfil clínico común que nadie definió de antemano—.

Usaremos **k-means**, el método más popular: le decimos **cuántos grupos (k)** buscar, coloca *k* centros, asigna cada paciente al más cercano, recalcula cada centro como la **media** de sus pacientes y repite hasta estabilizarse. La regla del juego de hoy: **agruparemos usando solo las variables clínicas**, dejando *deliberadamente fuera* `evento_cv` y `riesgo_cv_10a`. El algoritmo **no sabe** qué es "riesgo": solo agrupa por parecido.


### 1.1 Conoce tus datos y elige las variables (sin la etiqueta)

Abrimos `pacientes.csv` y nos quedamos con las **variables clínicas numéricas** para medir el "parecido" entre pacientes. Apartamos a propósito las dos columnas objetivo: las guardaremos para **validar al final**, no para agrupar.


In [ ]:
import pandas as pd
import numpy as np

pacientes = pd.read_csv("pacientes.csv")
print("Filas y columnas:", pacientes.shape)

# Variables clínicas que SÍ usamos para agrupar (NO incluimos evento_cv ni riesgo_cv_10a)
num = ["edad", "imc", "ta_sistolica", "ta_diastolica", "glucemia",
       "colesterol_total", "hdl", "antecedentes_familiares", "diabetes"]
print("\nVariables que entran al clustering:")
print(num)
pacientes[num].head()

**Lo que vemos:** cada **fila** es un paciente y cada **columna**, un dato clínico suyo. Estas son las variables con las que mediremos la cercanía entre pacientes. Fíjate en que `evento_cv` y `riesgo_cv_10a` **no están en la lista**: las reservamos. Así el ejercicio es honesto —el algoritmo agrupará "a ciegas" respecto al desenlace—.


### 1.2 Estandarizar: el paso que no puedes saltarte

k-means mide **distancias**. Y aquí hay una trampa: la `glucemia` viaja en cientos y el `imc` en decenas. Sin corregirlo, la variable de números grandes **dominaría** la distancia y mandaría sobre el resto. La solución es **estandarizar**: dejar todas las variables en la misma escala (media 0, desviación 1), para que cada una pese lo justo.


In [ ]:
from sklearn.preprocessing import StandardScaler

Xn = StandardScaler().fit_transform(pacientes[num])
print("Forma de los datos estandarizados:", Xn.shape)
print("Media de cada variable (≈0):", Xn.mean(axis=0).round(2))
print("Desviación de cada variable (≈1):", Xn.std(axis=0).round(2))

**Lo que hemos hecho:** ahora todas las variables tienen **media ≈ 0 y desviación ≈ 1**. Ninguna parte con ventaja por el mero hecho de estar en unidades más grandes. Sobre esta versión estandarizada trabajará k-means. *(Nota: estandarizar es igual de imprescindible para el PCA que usaremos después.)*


### 1.3 ¿Cuántos grupos? El método del codo (inercia)

k-means exige **fijar *k* de antemano**, y en fenotipado casi nunca es obvio cuántos subgrupos "hay". Una ayuda clásica es el **método del codo**: probamos varios valores de *k* y medimos la **inercia** (lo compactos que quedan los grupos: menor = más apretados). La inercia **siempre baja** al añadir grupos, así que buscamos el **"codo"**: el punto a partir del cual añadir más grupos apenas mejora.


In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

ks = range(2, 9)
inercias = []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(Xn)
    inercias.append(km.inertia_)

plt.figure(figsize=(7, 4))
plt.plot(list(ks), inercias, "o-", color="#00AEC7")
plt.axvline(4, ls="--", color="#E4572E", alpha=0.7, label="k = 4")
plt.xlabel("Número de grupos (k)")
plt.ylabel("Inercia (menor = grupos más compactos)")
plt.title("Método del codo")
plt.legend(); plt.tight_layout(); plt.show()

**Lo que vemos:** la inercia cae con fuerza de *k*=2 a *k*=4 y **luego se aplana**: a partir de ahí, cada grupo nuevo aporta muy poca compacidad extra. Ese cambio de pendiente —el "codo"— sugiere que **hacia 4 grupos** hay un buen equilibrio entre detalle y simplicidad. Pero el codo a veces no es nítido; conviene una segunda opinión.


### 1.4 Segunda opinión: el coeficiente de silueta

El **coeficiente de silueta** mide cómo de bien encaja cada paciente en su grupo frente al grupo vecino más cercano (va de −1 a 1; **más alto = grupos mejor definidos**). Lo calculamos para varios *k* y buscamos el máximo. *(Lo estimamos sobre una muestra para que sea rápido con 20 000 pacientes.)*


In [ ]:
from sklearn.metrics import silhouette_score

siluetas = []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(Xn)
    s = silhouette_score(Xn, km.labels_, sample_size=4000, random_state=0)
    siluetas.append(s)

for k, s in zip(ks, siluetas):
    marca = "  <- máximo" if s == max(siluetas) else ""
    print(f"k={k}:  silueta = {s:.3f}{marca}")

**Lo que significa:** la silueta es **más alta en *k*=4**. Las dos pistas coinciden: el codo apuntaba a 4 y la silueta lo confirma. **Decisión: *k*=4.**

Pero digámoslo claro, como manda esta unidad: estas métricas **orientan, no deciden**. La *k* final se apoya siempre en el **sentido clínico** —cuántos fenotipos son *accionables* e interpretables—. Cuatro grupos es, además, un número **manejable** para nombrarlos y usarlos.


### 1.5 Ajustamos k-means con k = 4

Ya podemos entrenar el modelo definitivo con 4 grupos y **etiquetar** a cada paciente con su clúster.


In [ ]:
km4 = KMeans(n_clusters=4, n_init=10, random_state=0).fit(Xn)
pacientes["cluster"] = km4.labels_

print("Nº de pacientes en cada clúster:")
print(pacientes["cluster"].value_counts().sort_index())

**Lo que vemos:** k-means ha repartido a los 20 000 pacientes en 4 grupos de **tamaños distintos** (unos más numerosos que otros). Eso es normal y esperable. Pero un reparto **no es un hallazgo**: falta lo esencial, que es entender **qué caracteriza** a cada grupo.


### 1.6 Caracterizar cada grupo: la tabla de perfiles medios

Aquí está el corazón del fenotipado. Calculamos, para cada clúster, la **media de cada variable clínica**: ese "perfil medio" es lo que nos deja **nombrar** los grupos. Para leerlos con orden, los presentamos **ordenados por edad media** (del más joven al más mayor) y les damos una etiqueta neutra `Fenotipo 1..4`.

**Importante:** hacemos esto **mirando solo las variables clínicas** —seguimos sin tocar `evento_cv`—.


In [ ]:
# Orden estable de los clústeres por edad media (del más joven al más mayor)
medias_cluster = pacientes.groupby("cluster")[num].mean()
orden = medias_cluster["edad"].sort_values().index
mapa = {cl: f"Fenotipo {i+1}" for i, cl in enumerate(orden)}
pacientes["fenotipo"] = pacientes["cluster"].map(mapa)

# Tabla de perfiles medios, ordenada y legible
perfil = pacientes.groupby("fenotipo")[num].mean().reindex([f"Fenotipo {i+1}" for i in range(4)])
perfil.insert(0, "n_pacientes", pacientes["fenotipo"].value_counts().reindex(perfil.index))
perfil.round(1)

**Cómo se lee esta tabla (cada fila, un fenotipo candidato):**

- **Fenotipo 1 — "joven, perfil favorable":** el más joven (unos **40 años**), IMC en rango normal, tensión y glucemia bajas, **HDL alto**. Sin diabéticos. Perfil metabólico sano.
- **Fenotipo 2 — "adulto intermedio":** edad media (**hacia los 48**), constantes en valores intermedios. Un grupo "de transición".
- **Fenotipo 3 — "mayor con tensión alta":** más mayor (unos **60 años**) y con **tensión sistólica elevada** (en torno a 135 mmHg). Perfil hipertensivo.
- **Fenotipo 4 — "perfil metabólico desfavorable":** mayor, **glucemia claramente alta** y **prácticamente todos diabéticos** (`diabetes` ≈ 1). Es el perfil más comprometido.

Fíjate en lo que ha ocurrido: **sin decirle nada al algoritmo**, ha separado a la población en gradientes clínicos reconocibles (edad, tensión, metabolismo). Eso es fenotipar. Pero **ojo**: estos nombres los ponemos *nosotros*; el algoritmo solo dibujó las fronteras.


### 1.7 Dibujar los fenotipos con PCA (2D)

Un paciente vive en un espacio de **9 dimensiones** que no podemos ver. El **PCA** (Análisis de Componentes Principales) resume esas 9 variables en **2 "ejes"** (componentes) que capturan la mayor parte de la variabilidad, y así podemos **dibujar** la cohorte en un plano. Si los fenotipos son reales, deberían verse como **zonas separadas**.


In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
coords = pca.fit_transform(Xn)   # dos coordenadas por paciente
var = pca.explained_variance_ratio_
print(f"Varianza explicada -> Componente 1: {var[0]:.1%} | Componente 2: {var[1]:.1%} | Juntas: {var.sum():.1%}")

**Lo que significa:** las dos primeras componentes retienen en torno al **46 %** de la información —casi la mitad de la variabilidad de las 9 variables, comprimida en 2 ejes—. No es todo (por eso algunos grupos se verán algo solapados), pero es **suficiente para asomarnos** a la estructura. Ahora pintamos cada paciente con el color de su fenotipo.


In [ ]:
colores = {"Fenotipo 1": "#00AEC7", "Fenotipo 2": "#2E9E5B",
           "Fenotipo 3": "#E4572E", "Fenotipo 4": "#7E5B9E"}

plt.figure(figsize=(7.5, 6))
for nombre, color in colores.items():
    m = pacientes["fenotipo"] == nombre
    plt.scatter(coords[m.values, 0], coords[m.values, 1],
                s=6, alpha=0.3, color=color, label=nombre)
plt.xlabel(f"Componente principal 1 ({var[0]:.0%} de la variabilidad)")
plt.ylabel(f"Componente principal 2 ({var[1]:.0%} de la variabilidad)")
plt.title("Los fenotipos, proyectados a 2D con PCA")
leg = plt.legend(markerscale=3);
plt.tight_layout(); plt.show()

**Lo que vemos:** los cuatro fenotipos ocupan **regiones distintas** del plano —se aprecia un gradiente de izquierda a derecha—, aunque **no están perfectamente separados**: hay zonas donde se rozan. Es lo esperable, porque estamos viendo solo el 46 % de la información y porque la población clínica es en buena parte un **continuo**, no cuatro islas. Que se distingan regiones es un **buen indicio**; que se solaparan del todo, sería una señal para dudar de que fueran fenotipos distintos.


### 1.8 La validación honesta: ¿y el evento que apartamos?

Llega el momento más instructivo. Formamos los grupos **sin** `evento_cv`. Ahora **miramos** cómo se reparte ese desenlace que habíamos guardado. Si los fenotipos capturan algo clínicamente real, deberían tener **tasas de evento distintas**.


In [ ]:
tasa_evento = (pacientes.groupby("fenotipo")["evento_cv"].mean()
               .reindex([f"Fenotipo {i+1}" for i in range(4)]))

plt.figure(figsize=(7, 4))
plt.bar(tasa_evento.index, tasa_evento.values * 100,
        color=[colores[f] for f in tasa_evento.index], edgecolor="white")
for i, v in enumerate(tasa_evento.values):
    plt.text(i, v * 100 + 0.4, f"{v:.1%}", ha="center", fontweight="bold")
plt.ylabel("Eventos cardiovasculares (%)")
plt.title("Tasa de evento_cv por fenotipo  (¡NO se usó para agrupar!)")
plt.tight_layout(); plt.show()

print(tasa_evento.apply(lambda x: f"{x:.1%}"))

**Lo que significa —y es notable:** la tasa de eventos **crece de forma ordenada** del Fenotipo 1 al 4 (de un **~5 %** en el grupo joven-favorable hasta un **~40 %** en el grupo metabólico desfavorable). Y esto lo hemos obtenido **sin que el algoritmo viera nunca el evento**: agrupó solo por parecido clínico, y aun así los grupos ordenan el riesgo. Eso **valida** que los fenotipos capturan estructura clínica real.

**Cuidado con la conclusión, sin embargo:** esto **valida** la interpretación, **no convierte el ejercicio en supervisado**. No hemos "predicho" el evento; hemos comprobado que unos grupos definidos a ciegas resultan tener sentido clínico. Es un truco honesto y revelador, no un clasificador.


{% hint style="warning" %}
**⚠️ Aviso · No leer significado donde solo hay ruido (el error más caro de esta unidad)**

k-means **siempre** devuelve *k* grupos, aunque los datos sean puro azar. Por eso el peligro nº 1 del fenotipado es la **sobre-interpretación**: mirar un clúster, ponerle una etiqueta clínica sugerente y creer que hemos *descubierto* una realidad, cuando quizá solo hemos partido una nube continua por una línea arbitraria.

Defensas concretas: (1) desconfía de grupos que **no se sostienen** al cambiar la semilla, las variables o el algoritmo —un fenotipo real es **estable**—; (2) exige que el grupo tenga un **perfil interpretable** y, a ser posible, que se replique en **otra cohorte**; (3) recuerda que una silueta alta indica grupos *compactos*, no grupos *clínicamente reales*. Un subgrupo estadísticamente nítido que ningún clínico sabe nombrar es, casi siempre, una **alarma**, no un hallazgo.
{% endhint %}


## 2. Segmentar centros por su perfil de actividad

La misma idea, cambiando de sujeto. Sobre `centros.csv` agrupamos los **centros sanitarios** por su perfil de actividad, sin ninguna etiqueta. El objetivo clínico-gestor: **comparar centros parecidos entre sí** (benchmarking justo —no medir un ambulatorio con la vara de un hospital—) y **planificar recursos por perfil**.


### 2.1 Conoce los centros

Abrimos la tabla y miramos las cuatro variables de perfil que pide el análisis: **camas**, número de **servicios**, **urgencias diarias medias** y **proporción de mayores de 65**.


In [ ]:
centros = pd.read_csv("centros.csv")
print("Centros:", len(centros))
feat_c = ["camas", "n_servicios", "urgencias_dia_media", "ratio_mayores65"]
centros[["centro_id"] + feat_c].head(8)

**Lo que vemos:** unos pocos centros con **muchísimas camas y servicios** conviven con muchos otros **pequeños**. A ojo ya se intuye que no todos juegan en la misma liga; el clustering lo hará explícito y objetivo.


### 2.2 Estandarizar y elegir cuántos grupos

Igual que con los pacientes, **estandarizamos** (las camas van en cientos y `ratio_mayores65` entre 0 y 1: sin corregir escala, las camas mandarían). Y usamos la **silueta** para decidir cuántos grupos hay.


In [ ]:
Xc = StandardScaler().fit_transform(centros[feat_c])

for k in range(2, 6):
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(Xc)
    s = silhouette_score(Xc, km.labels_)
    marca = "  <- máximo" if k == 2 else ""
    print(f"k={k}:  silueta = {s:.3f}{marca}")

**Lo que significa:** la silueta es **claramente más alta en *k*=2** (bastante por encima del resto). Los datos "quieren" partirse en **dos grupos** muy bien separados. Añadir más grupos solo trocea uno de los dos en subgrupos casi idénticos —división que **no sería accionable**—. Nos quedamos con **2 grupos**.


### 2.3 Ajustamos y describimos los dos perfiles


In [ ]:
km_c = KMeans(n_clusters=2, n_init=10, random_state=0).fit(Xc)
centros["grupo"] = km_c.labels_

# Ordenamos los grupos por tamaño (nº de camas) para leerlos: pequeño vs grande
perfil_c = centros.groupby("grupo")[feat_c].mean()
orden_c = perfil_c["camas"].sort_values().index
etiqueta = {orden_c[0]: "Perfil A (pequeño)", orden_c[1]: "Perfil B (grande)"}
centros["perfil"] = centros["grupo"].map(etiqueta)

resumen_c = centros.groupby("perfil")[feat_c].mean().round(1)
resumen_c.insert(0, "n_centros", centros["perfil"].value_counts())
resumen_c

**Cómo se leen los dos perfiles:**

- **Perfil A (pequeño):** muchos centros con **pocas camas**, **pocos servicios** y **baja presión de urgencias**. Encaja con el retrato de un **centro de salud / ambulatorio**.
- **Perfil B (grande):** unos pocos centros con **cientos de camas**, **muchos servicios** y **alta actividad de urgencias**. El retrato de un **hospital**.

La proporción de mayores de 65 es parecida en ambos: **lo que separa a los centros es el tamaño y la actividad**, no el envejecimiento de su población. Esta segmentación permite comparar cada centro **con los de su liga** y diseñar políticas por perfil en vez de una talla única.


### 2.4 Validación honesta (otra vez): ¿coincide con el tipo de centro?

`centros.csv` incluye una columna `tipo` (hospital / centro de salud) que **no usamos** para agrupar. Es nuestra "etiqueta reservada", como el `evento_cv` de antes. Crucemos el perfil que **descubrió** el algoritmo con el tipo real.


In [ ]:
pd.crosstab(centros["perfil"], centros["tipo"])

**Lo que vemos:** la correspondencia es **casi perfecta** —el "Perfil A (pequeño)" agrupa a los centros de salud y el "Perfil B (grande)", a los hospitales—, y lo ha logrado mirando **solo** camas, servicios, urgencias y proporción de mayores, **sin** que le dijéramos el tipo. Igual que con los pacientes: un agrupamiento a ciegas que, al validarlo contra una etiqueta apartada, resulta tener **pleno sentido**. Reconfortante… y una buena señal de que la estructura era real.


## 3. Detección de anomalías en wearables

Detectar lo que se sale de lo normal es uno de los usos más rentables del no supervisado: las **anomalías** —errores de registro, eventos raros— suelen ser **escasas y sin etiqueta**. La idea es **aprender cómo es "lo normal" y señalar lo que se desvía**.

`wearable.csv` recoge, por **sujeto y día**, la **frecuencia cardiaca en reposo** (`fc_reposo`), los **pasos** y las **horas de sueño**. Buscaremos los **sujetos-día raros**.


### 3.1 Conoce la señal

Miramos la tabla y un resumen numérico para saber qué es "normal" en estos wearables.


In [ ]:
wear = pd.read_csv("wearable.csv")
print("Registros (sujeto-día):", len(wear), "| Sujetos:", wear["sujeto_id"].nunique())
print()
print(wear[["fc_reposo", "pasos", "horas_sueno"]].describe().round(1))
wear.head()

**Lo que vemos:** cada fila es **un día de un sujeto**. La frecuencia cardiaca en reposo ronda las **60-70 pulsaciones**, con miles de pasos y unas **7 horas de sueño** de media. Sobre este "comportamiento habitual" el modelo aprenderá qué es normal para poder marcar lo que no encaja.


### 3.2 Isolation Forest: aislar lo raro

El **Isolation Forest** funciona con un giro elegante: en lugar de modelar lo normal con detalle, **aísla** los puntos con cortes al azar. Las anomalías, al ser distintas, **quedan separadas con muy pocos cortes**; un punto normal, rodeado de parecidos, necesita muchos más. Cuanto antes se aísla un caso, más anómalo es.

Le indicamos con `contamination` qué **fracción** de casos esperamos que sean anómalos (aquí un **2 %**): es un umbral que fijamos con criterio, no una verdad.


In [ ]:
from sklearn.ensemble import IsolationForest

Xw = StandardScaler().fit_transform(wear[["fc_reposo", "pasos", "horas_sueno"]])
iso = IsolationForest(contamination=0.02, random_state=0)
wear["anomalo"] = iso.fit_predict(Xw) == -1   # el modelo marca -1 = anómalo

print("Sujetos-día marcados como anómalos:", int(wear["anomalo"].sum()), "de", len(wear))
print(f"({wear['anomalo'].mean():.1%} del total, como pedimos con contamination=0.02)")

**Lo que significa:** el modelo ha señalado un **2 % de sujetos-día** como atípicos. No dice que estén "mal" ni que estén "enfermos": dice que su **combinación** de frecuencia cardiaca, pasos y sueño **no encaja con lo habitual**. Ahora toca lo importante —mirar *qué* tienen de raro—.


### 3.3 ¿Qué caracteriza a los días anómalos?

Comparamos el perfil medio de los días **normales** frente a los **anómalos**.


In [ ]:
comparacion = wear.groupby("anomalo")[["fc_reposo", "pasos", "horas_sueno"]].mean().round(1)
comparacion.index = ["Días normales", "Días anómalos"]
comparacion

**Lo que vemos:** los días anómalos destacan sobre todo por una **frecuencia cardiaca en reposo más alta** y **menos horas de sueño** que lo habitual. Es exactamente el tipo de patrón —el corazón acelerado en reposo, el sueño acortado— que en la vida real haría **sospechar que algo pasa** ese día.


### 3.4 Veamos algunos casos concretos

Nada como mirar los ejemplos. Ordenamos por frecuencia cardiaca y mostramos los sujetos-día más llamativos.


In [ ]:
ejemplos = (wear[wear["anomalo"]]
            .sort_values("fc_reposo", ascending=False)
            .head(8)[["sujeto_id", "dia", "fc_reposo", "pasos", "horas_sueno"]])
ejemplos.reset_index(drop=True)

**Lo que vemos:** varios sujetos con la **frecuencia cardiaca en reposo disparada** (muy por encima de sus ~65 habituales) y el **sueño reducido**. Y un detalle revelador: muchos de estos episodios **se concentran en días consecutivos del mismo sujeto** —como si atravesara una racha de días "raros"—, justo lo que esperaríamos de un proceso real (una infección, unos días de fiebre) y no de un fallo puntual aislado.


### 3.5 La doble lectura clínica (y por qué el algoritmo no decide)

Una anomalía admite **dos lecturas muy distintas**, y distinguirlas es el trabajo del profesional:

- **Error de dato / artefacto del sensor.** El caso es raro porque está **mal medido**: una mala colocación de la pulsera, un mal contacto. Es la versión **automática y multivariante** del control de calidad de la U2: puede cazar combinaciones rarísimas *aunque cada valor por separado sea plausible* —algo que un simple filtro de rangos no vería—.
- **Evento clínico auténtico.** El caso es raro porque **de verdad** lo es: fiebre, una infección incipiente, una arritmia, el efecto de un fármaco. Lejos de ser basura, puede ser el sujeto **más interesante** —el que merece una llamada—.

El modelo **no sabe** cuál de las dos es. Solo dice *"esto no encaja con lo normal de esta persona"*. Quién decide si es ruido o señal es el equipo clínico.


{% hint style="warning" %}
**⚠️ Aviso · Validar sin etiquetas es el verdadero reto (human-in-the-loop)**

Como no hay verdad de referencia, **decidir si una "anomalía" es realmente un problema exige revisión humana y conocimiento clínico**. La práctica correcta no es que el sistema decida solo, sino que **priorice** los casos más sospechosos para que una persona los revise: el algoritmo pone el foco, el clínico juzga. Y no hay que **sobre-interpretar**: el Isolation Forest marca lo **raro**, no lo **malo** —rareza no es enfermedad ni error por sí sola—.
{% endhint %}


## 4. Qué hemos aprendido

- **Sin etiqueta, la interpretación clínica manda.** El clustering *propone* fenotipos y el Isolation Forest *marca* casos raros, pero **quién decide qué significan es el profesional**. Aquí el criterio no es un añadido: es el núcleo.
- **Fenotipar es agrupar por parecido.** Con *k-means* sobre `pacientes.csv` (usando **solo** variables clínicas, estandarizadas) surgieron **4 fenotipos** con perfiles reconocibles, del joven-favorable al metabólico-desfavorable. Al cruzarlos *a posteriori* con el `evento_cv` reservado, la tasa de eventos crecía de forma ordenada (~5 % → ~40 %): eso **valida** la interpretación, sin convertir el ejercicio en supervisado.
- **El PCA es una lente para ver.** Resumió 9 variables en 2 componentes (~46 % de la variabilidad) y nos dejó **dibujar** los fenotipos y comprobar que ocupan regiones distintas del plano.
- **La misma idea segmenta centros.** Por camas, servicios, urgencias y proporción de mayores, `centros.csv` se partió en **2 perfiles** (pequeño vs grande) que coincidían casi perfectamente con el tipo real de centro —otra validación honesta contra una etiqueta apartada—.
- **Las anomalías tienen doble cara.** Un sujeto-día raro puede ser un **error de sensor** o un **evento clínico auténtico**. El Isolation Forest **prioriza** los sospechosos; el clínico decide —*human-in-the-loop*—.
- **El peligro nº 1: sobre-interpretar.** k-means siempre da *k* grupos y una anomalía siempre se puede marcar. Un fenotipo solo es creíble si es **estable, interpretable** y, a ser posible, **replicable**. Un clúster que nadie sabe nombrar es una alarma, no un hallazgo.

**Puente a la U7:** hasta aquí hemos tratado los datos como una tabla "plana": cada fila, un paciente o un centro independiente. Pero mucha información clínica tiene una dimensión que hemos dejado para su sitio —el **tiempo**—. Los ingresos en urgencias son **series temporales**, y predecirlas bien exige cuidados propios (empezando por no dejar que el futuro se cuele en el pasado). Ese es el territorio de la **U7**.


## 5. Y esto, ¿cómo se lo pido a un asistente de IA?

Recuerda el principio del curso: **tú pones el criterio, la IA escribe el código.** En no supervisado, lo que marca la diferencia no es el código —lo escribe el agente— sino **cuánto contexto le das**. Un buen encargo para reproducir este cuaderno sería:

> *"Actúa como científico de datos clínicos senior. Con `pacientes.csv` (datos **sintéticos**), en español y por celdas: quiero **descubrir fenotipos** de paciente. Usa **solo** las variables clínicas (edad, IMC, tensión, glucemia, colesterol, HDL, antecedentes, diabetes); **no uses** `evento_cv` ni `riesgo_cv_10a` para agrupar. (1) EDA breve y **estandariza** (justifica por qué). (2) Elige *k* con **método del codo** y **silueta**, ajusta *k*-means y **nombra cada fenotipo** por su perfil medio. (3) **Dibuja** los grupos con **PCA** en 2D. (4) **Solo al final**, cruza cada fenotipo con `evento_cv` para ver si alguno concentra más eventos —eso **valida**, no sustituye—. (5) Marca pacientes **anómalos** con Isolation Forest. Al terminar, **evalúa críticamente**: ¿son grupos estables e interpretables o artefactos? Si no convence, propón y ejecuta una segunda versión. Itera 2-3 veces."*

Tu trabajo con su respuesta es el de siempre: comprobar que **no se usó la etiqueta** para agrupar, que los grupos son **interpretables y estables**, y que la validación *a posteriori* se presenta como lo que es —una comprobación, no una predicción—. Ese criterio es lo que no se automatiza.
